In [ ]:
!pip install -q transformers

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, confusion_matrix
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (6,4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/triagegeist/train.csv")
test = pd.read_csv("/kaggle/input/competitions/triagegeist/test.csv")
history = pd.read_csv("/kaggle/input/competitions/triagegeist/patient_history.csv")
complaints = pd.read_csv("/kaggle/input/competitions/triagegeist/chief_complaints.csv")

train = train.merge(history, on="patient_id", how="left")
test = test.merge(history, on="patient_id", how="left")

train = train.merge(complaints, on="patient_id", how="left")
test = test.merge(complaints, on="patient_id", how="left")

In [ ]:
plt.figure()
train["triage_acuity"].value_counts().sort_index().plot(kind="bar")
plt.title("Triage Acuity Distribution")
plt.xlabel("Acuity Level")
plt.ylabel("Count")
plt.show()

In [ ]:
for col in ["disposition", "ed_los_hours"]:
    if col in train.columns:
        train.drop(columns=col, inplace=True)

In [ ]:
def feature_engineering(df):
    df["shock_index"] = df["heart_rate"] / (df["systolic_bp"] + 1e-6)
    df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]
    df["map"] = (2 * df["diastolic_bp"] + df["systolic_bp"]) / 3
    return df

train = feature_engineering(train)
test = feature_engineering(test)

In [ ]:
train["pain_score"] = train["pain_score"].replace(-1, np.nan)
test["pain_score"] = test["pain_score"].replace(-1, np.nan)

common_numeric_cols = list(
    set(train.select_dtypes(include=np.number).columns)
    .intersection(set(test.select_dtypes(include=np.number).columns))
)

if "triage_acuity" in common_numeric_cols:
    common_numeric_cols.remove("triage_acuity")

for col in common_numeric_cols:
    median = train[col].median()
    train[col] = train[col].fillna(median)
    test[col] = test[col].fillna(median)

In [ ]:
categorical_cols = train.select_dtypes(include="object").columns.tolist()
categorical_cols.remove("patient_id")
categorical_cols.remove("chief_complaint_raw")

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

target = "triage_acuity"

features = [col for col in train.columns if col not in ["patient_id", target, "chief_complaint_raw"]]

numeric_cols = train[features].select_dtypes(include=np.number).columns.tolist()

scaler = StandardScaler()
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])
test[numeric_cols] = scaler.transform(test[numeric_cols])

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
class TriageDataset(Dataset):
    def __init__(self, df, features, tokenizer, target=None, max_len=64):
        self.X = df[features].values.astype(np.float32)
        self.texts = df["chief_complaint_raw"].fillna("").tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.y = df[target].values - 1 if target else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "tabular": torch.tensor(self.X[idx]),
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

        if self.y is not None:
            item["label"] = torch.tensor(self.y[idx])

        return item

In [ ]:
class TriageNetMulti(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()

        self.tabular_net = nn.Sequential(
            nn.Linear(tabular_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.text_fc = nn.Linear(768, 128)

        self.fusion = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 5)
        )

    def forward(self, tabular, input_ids, attention_mask):
        tab_out = self.tabular_net(tabular)
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = bert_out.last_hidden_state[:, 0, :]
        text_out = self.text_fc(cls_token)
        combined = torch.cat([tab_out, text_out], dim=1)
        return self.fusion(combined)

In [ ]:
FOLDS = 3
EPOCHS = 2
BATCH_SIZE = 64
LR = 2e-5

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[target])):

    print(f"\n===== Fold {fold+1} =====")

    train_df = train.iloc[train_idx]
    val_df = train.iloc[val_idx]

    train_dataset = TriageDataset(train_df, features, tokenizer, target)
    val_dataset = TriageDataset(val_df, features, tokenizer, target)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = TriageNetMulti(len(features)).to(device)

    class_counts = train_df[target].value_counts().sort_index().values
    class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    fold_loss_history = []

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader):
            optimizer.zero_grad()

            tab = batch["tabular"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(tab, ids, mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        fold_loss_history.append(avg_loss)

        print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

    # Plot fold loss
    plt.figure()
    plt.plot(fold_loss_history)
    plt.title(f"Fold {fold+1} Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

    # Validation predictions
    model.eval()
    val_preds = []

    with torch.no_grad():
        for batch in val_loader:
            tab = batch["tabular"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            outputs = model(tab, ids, mask)
            preds = torch.argmax(outputs, dim=1)

            val_preds.extend(preds.cpu().numpy())

    oof_preds[val_idx] = val_preds

In [ ]:
true_labels = train[target].values - 1

macro_f1 = f1_score(true_labels, oof_preds, average="macro")

print("Final OOF Macro F1:", macro_f1)

In [ ]:
cm = confusion_matrix(true_labels, oof_preds)

plt.figure()
plt.imshow(cm)
plt.colorbar()
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()